# NUS DSS5103 — Geospatial Data Analysis
## Detailed, systematic, student-intuitive case study with Python + Bokeh

This notebook is designed as a **teaching notebook**, not just a set of code snippets.

We use one coherent sustainability case study:

> **Urban heat exposure:** Which places are hotter, are hot areas spatially clustered, what factors explain the pattern, how do neighbouring areas matter, and how can we estimate temperature where no sensor exists?

The notebook progresses methodically:

1. geospatial thinking and data models;
2. vector geometry and raster intuition;
3. coordinate reference systems (CRS);
4. spatial joins, buffers and nearest-neighbour operations;
5. Bokeh choropleths and exploratory spatial data analysis;
6. spatial weights and adjacency graphs;
7. global Moran's $I$;
8. Local Moran / LISA hot- and cold-spots;
9. OLS baseline and residual spatial diagnostics;
10. spatial lag and spatial error models;
11. point-pattern KDE;
12. IDW interpolation;
13. empirical variograms;
14. spatial block cross-validation;
15. spatiotemporal extension;
16. raster thinking, spatial indexing, scale and MAUP;
17. student experiments and advanced extensions.

### Learning philosophy

The core idea is:

$$
\boxed{
\text{location is part of the statistical structure, not merely metadata}
}
$$

Nearby observations can share environment, infrastructure, exposure and latent causes. That means conventional assumptions such as independent residuals may fail.

### Why a reproducible case study?

Instead of depending on a brittle external download URL, we construct a **Singapore-inspired spatial dataset** whose generating mechanism is known. That lets us compare what each method discovers against the underlying process.

The notebook still uses real geospatial tooling:

- GeoPandas / Shapely;
- PySAL (`libpysal`, `esda`, `spreg`);
- SciPy / scikit-learn;
- NetworkX;
- Bokeh for all major visualisations.

# 0. Environment setup

Install the packages used in the notebook. If your environment already contains them, pip will reuse the installed versions.

> If Jupyter asks for a kernel restart after installation, restart once and continue from the import cell.

In [1]:
#%uv pip install -q numpy pandas scipy scikit-learn bokeh geopandas shapely pyproj libpysal esda spreg networkx

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Sequence, Tuple
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, box
from scipy.spatial.distance import cdist
from scipy.stats import gaussian_kde

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

import networkx as nx

from bokeh.io import output_notebook, show
from bokeh.layouts import row, column
from bokeh.models import ColumnDataSource, ColorBar, HoverTool, LinearColorMapper, Span
from bokeh.palettes import Viridis256, Turbo256, Category10
from bokeh.plotting import figure

output_notebook()

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
warnings.filterwarnings('ignore', category=FutureWarning)

print('GeoPandas version:', gpd.__version__)

Loading BokehJS ...

GeoPandas version: 1.1.4


# 1. Geospatial thinking: why location changes the problem

A conventional observation is often represented as

$$
(x_1,x_2,\ldots,x_p,y).
$$

A geospatial observation additionally has geometry:

$$
(\text{geometry},x_1,x_2,\ldots,x_p,y).
$$

For our case study, one row represents one urban grid cell.

| Field | Interpretation |
|---|---|
| `geometry` | polygon representing an urban area |
| `vegetation` | fraction of green cover |
| `building_density` | built-up intensity |
| `road_density` | proxy for impervious surface / transport intensity |
| `distance_to_coast_km` | coastal proximity |
| `population_density` | people potentially exposed |
| `temperature_c` | land-surface temperature |

The key complication is **spatial dependence**.

Ordinary regression commonly relies on residual independence:

$$
\operatorname{Cov}(\epsilon_i,\epsilon_j)=0,\quad i\neq j.
$$

But neighbouring districts can share microclimate, land cover, building morphology and traffic. Therefore

$$
\operatorname{Cov}(\epsilon_i,\epsilon_j)\neq 0
$$

may be more realistic.

This is why spatial data science needs explicit ideas of **distance, adjacency, neighbourhood and scale**.

# 2. Reusable configuration

Centralising experimental parameters makes the notebook easier to reproduce and extend.

In [3]:
@dataclass(frozen=True)
class SpatialCaseConfig:
    n_cols: int = 22
    n_rows: int = 14
    cell_size_m: float = 1000.0
    crs: str = 'EPSG:3414'  # SVY21 / Singapore TM
    n_sensors: int = 85
    spatial_noise_strength: float = 1.7
    iid_noise_sd: float = 0.55
    random_state: int = 42

CFG = SpatialCaseConfig()
CFG

SpatialCaseConfig(n_cols=22, n_rows=14, cell_size_m=1000.0, crs='EPSG:3414', n_sensors=85, spatial_noise_strength=1.7, iid_noise_sd=0.55, random_state=42)

# 3. Vector geometry: points, lines and polygons

Three fundamental vector geometry types are:

- **Point** — weather sensor, tree, customer, charging station;
- **LineString** — road, river, MRT segment;
- **Polygon** — planning area, land parcel, building footprint.

We start with polygons arranged on a regular grid. The grid is intentionally simple because it makes adjacency and raster conversion easy to understand.

In [4]:
def build_grid(cfg: SpatialCaseConfig) -> gpd.GeoDataFrame:
    records = []
    for r in range(cfg.n_rows):
        for c in range(cfg.n_cols):
            x0 = c * cfg.cell_size_m
            y0 = r * cfg.cell_size_m
            geom = box(x0, y0, x0 + cfg.cell_size_m, y0 + cfg.cell_size_m)
            records.append({
                'cell_id': f'R{r:02d}_C{c:02d}',
                'row': r,
                'col': c,
                'geometry': geom,
            })
    return gpd.GeoDataFrame(records, crs=cfg.crs)

grid = build_grid(CFG)
display(grid.head())
print('Rows:', len(grid))
print('Geometry:', grid.geom_type.value_counts().to_dict())
print('CRS:', grid.crs)

,cell_id,row,col,geometry
0,R00_C00,0,0,"POLYGON ((1000 0, 1000 1000, 0 1000, 0 0, 1000..."
1,R00_C01,0,1,"POLYGON ((2000 0, 2000 1000, 1000 1000, 1000 0..."
2,R00_C02,0,2,"POLYGON ((3000 0, 3000 1000, 2000 1000, 2000 0..."
3,R00_C03,0,3,"POLYGON ((4000 0, 4000 1000, 3000 1000, 3000 0..."
4,R00_C04,0,4,"POLYGON ((5000 0, 5000 1000, 4000 1000, 4000 0..."


Rows: 308
Geometry: {'Polygon': 308}
CRS: EPSG:3414


## 3.1 Coordinate Reference Systems (CRS)

A CRS tells us what the geometry coordinates mean.

### Geographic CRS

Longitude/latitude coordinates are angular:

$$
(103.85^\circ,1.29^\circ).
$$

### Projected CRS

A projected CRS maps the curved Earth to a plane and often expresses coordinates in metres.

For operations such as distance, area and buffering, a suitable projected CRS is usually much safer.

This notebook uses **EPSG:3414 (SVY21 / Singapore TM)**.

A common mistake is computing distance while coordinates are still in degrees. The software may return a number, but its interpretation can be wrong.

In [5]:
grid['area_m2'] = grid.geometry.area
print('One grid cell area:', f"{grid['area_m2'].iloc[0]:,.0f}", 'm²')
print('One grid cell area:', f"{grid['area_m2'].iloc[0]/1_000_000:.2f}", 'km²')

One grid cell area: 1,000,000 m²
One grid cell area: 1.00 km²


# 4. Generate spatially structured sustainability features

We deliberately make the predictors vary smoothly over space:

- a dense urban core;
- green corridors;
- road intensity near the centre;
- population concentrated around built-up areas;
- a coastal-distance feature.

The important lesson is that **the covariates themselves can be spatially structured**.

In [6]:
def attach_centroids(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    out = gdf.copy()
    cent = out.geometry.centroid
    out['x_km'] = cent.x / 1000.0
    out['y_km'] = cent.y / 1000.0
    return out

grid = attach_centroids(grid)
x = grid['x_km'].to_numpy()
y = grid['y_km'].to_numpy()

x_norm = (x - x.min()) / (x.max() - x.min())
y_norm = (y - y.min()) / (y.max() - y.min())

urban_core = np.exp(-(((x_norm-0.52)/0.23)**2 + ((y_norm-0.48)/0.26)**2))
green_corridor = np.exp(-(((x_norm-0.23)/0.15)**2 + ((y_norm-0.68)/0.22)**2))
secondary_green = np.exp(-(((x_norm-0.78)/0.13)**2 + ((y_norm-0.30)/0.18)**2))

vegetation = 0.18 + 0.55*green_corridor + 0.35*secondary_green - 0.22*urban_core + rng.normal(0,0.04,len(grid))
vegetation = np.clip(vegetation, 0.03, 0.90)

building_density = 0.12 + 0.70*urban_core + 0.08*np.sin(3*np.pi*x_norm) + rng.normal(0,0.04,len(grid))
building_density = np.clip(building_density, 0.02, 0.95)

road_axis = np.exp(-((y_norm-0.50)/0.14)**2)
road_density = 0.15 + 0.52*urban_core + 0.18*road_axis + rng.normal(0,0.05,len(grid))
road_density = np.clip(road_density, 0.02, 1.00)

distance_to_coast_km = np.minimum.reduce([
    x - x.min() + 0.5,
    x.max() - x + 0.5,
    y - y.min() + 0.5,
    y.max() - y + 0.5,
])

population_density = 2500 + 15500*urban_core + 3000*building_density + rng.normal(0,900,len(grid))
population_density = np.clip(population_density, 500, None)

grid['vegetation'] = vegetation
grid['building_density'] = building_density
grid['road_density'] = road_density
grid['distance_to_coast_km'] = distance_to_coast_km
grid['population_density'] = population_density

display(grid[['cell_id','vegetation','building_density','road_density','distance_to_coast_km','population_density']].head())

,cell_id,vegetation,building_density,road_density,distance_to_coast_km,population_density
0,R00_C00,0.192149,0.143446,0.185795,0.5,2783.887354
1,R00_C01,0.138302,0.224344,0.162336,0.5,4874.544460
2,R00_C02,0.209795,0.230408,0.119870,0.5,3055.416205
3,R00_C03,0.217156,0.217132,0.223729,0.5,3558.676520
4,R00_C04,0.101060,0.270726,0.130178,0.5,3169.004067


# 5. Generate temperature with a latent spatial process

We simulate temperature as

$$
T_i = 29 - 2.8V_i + 2.5B_i + 1.4R_i + 0.13D_i + S_i + \epsilon_i.
$$

where $S_i$ is a latent smooth spatial effect.

Interpretation of the signs:

- vegetation cools;
- denser buildings and roads heat;
- being farther from the coast raises temperature slightly;
- an unobserved spatial field captures omitted geographically structured influences.

This is important because OLS will not be given $S_i$. We can later ask whether residual diagnostics discover the missing spatial structure.

In [7]:
spatial_field = (
    np.sin(2*np.pi*x_norm)
    + 0.8*np.cos(2.3*np.pi*y_norm)
    + 0.5*np.sin(np.pi*(x_norm+y_norm))
)
spatial_field = CFG.spatial_noise_strength * spatial_field / np.std(spatial_field)
iid_noise = rng.normal(0, CFG.iid_noise_sd, len(grid))

grid['latent_spatial_effect'] = spatial_field
grid['temperature_c'] = (
    29.0
    - 2.8*grid['vegetation'].to_numpy()
    + 2.5*grid['building_density'].to_numpy()
    + 1.4*grid['road_density'].to_numpy()
    + 0.13*grid['distance_to_coast_km'].to_numpy()
    + spatial_field
    + iid_noise
)

grid['heat_exposure_index'] = grid['temperature_c'] * np.log1p(grid['population_density'])
display(grid[['temperature_c','vegetation','building_density','road_density','population_density']].describe().round(3))

,temperature_c,vegetation,building_density,road_density,population_density
count,308.000,308.000,308.000,308.000,308.000
mean,29.968,0.216,0.251,0.275,5829.779
std,1.654,0.129,0.150,0.173,4129.272
min,25.062,0.030,0.020,0.044,500.000
25%,28.750,0.139,0.154,0.155,3207.220
50%,30.014,0.191,0.214,0.221,4266.181
75%,31.236,0.258,0.295,0.335,6749.411
max,33.668,0.727,0.798,0.877,20864.569


# 6. Reusable Bokeh choropleth utilities

A choropleth maps a numerical variable to polygon fill intensity.

We extract polygon exterior coordinates and pass them to Bokeh `patches`.

In [41]:
def polygon_xy(geom: Polygon) -> Tuple[list, list]:
    xs, ys = geom.exterior.xy
    return list(xs), list(ys)


def geodata_source(gdf: gpd.GeoDataFrame, extra_cols: Optional[Sequence[str]] = None) -> ColumnDataSource:
    extra_cols = list(extra_cols or [])
    xs, ys = [], []
    for geom in gdf.geometry:
        xvals, yvals = polygon_xy(geom)
        xs.append(xvals)
        ys.append(yvals)
    data = {
        'xs': xs,
        'ys': ys,
        'cell_id': gdf['cell_id'].astype(str).tolist(),
    }
    for col in extra_cols:
        data[col] = gdf[col].tolist()
    return ColumnDataSource(data)


def choropleth(gdf, value_col, title, palette=Viridis256, width=800, height=470):
    src = geodata_source(gdf, [value_col])
    mapper = LinearColorMapper(
        palette=palette,
        low=float(gdf[value_col].min()),
        high=float(gdf[value_col].max()),
    )
    p = figure(
        width=width,
        height=height,
        title=title,
        match_aspect=True,
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        x_axis_label='SVY21 Easting (m)',
        y_axis_label='SVY21 Northing (m)',
    )
    r = p.patches(
        xs='xs', ys='ys', source=src,
        fill_color={'field': value_col, 'transform': mapper},
        fill_alpha=0.90, line_alpha=0.30, line_width=0.5,
    )
    p.add_tools(HoverTool(renderers=[r], tooltips=[
        ('cell','@cell_id'),
        (value_col, f'@{value_col}{{0.000}}'),
    ]))
    p.add_layout(ColorBar(color_mapper=mapper, width=12), 'right')
    return p

In [42]:
show(choropleth(grid, 'temperature_c', 'Simulated urban land-surface temperature', palette=Turbo256))

## 6.1 ESDA: ask questions before fitting models

From the map, ask:

1. Are hot cells spatially clustered?
2. Are cool zones aligned with vegetation?
3. Is the centre different from the boundary?
4. Are the hottest cells also the most populated?
5. Could a random train/test split place near-neighbours on opposite sides of the split?

This is **Exploratory Spatial Data Analysis (ESDA)**.

In [10]:
show(row(
    choropleth(grid, 'vegetation', 'Vegetation fraction', width=540, height=360),
    choropleth(grid, 'building_density', 'Building density', width=540, height=360),
))

# 7. Spatial joins

A relational join uses keys:

```sql
ON left.customer_id = right.customer_id
```

A spatial join uses a geometric predicate:

- point `within` polygon;
- polygon `contains` point;
- two geometries `intersect`;
- polygons `touch`.

We generate sensor points and join each sensor to the grid polygon containing it.

In [11]:
def sample_sensors(grid_gdf: gpd.GeoDataFrame, cfg: SpatialCaseConfig) -> gpd.GeoDataFrame:
    local_rng = np.random.default_rng(cfg.random_state + 1)
    sampled_idx = local_rng.choice(grid_gdf.index.to_numpy(), size=cfg.n_sensors, replace=False)
    rows = []
    for sensor_id, idx in enumerate(sampled_idx):
        poly = grid_gdf.loc[idx, 'geometry']
        minx, miny, maxx, maxy = poly.bounds
        px = local_rng.uniform(minx + 100, maxx - 100)
        py = local_rng.uniform(miny + 100, maxy - 100)
        observed = grid_gdf.loc[idx, 'temperature_c'] + local_rng.normal(0,0.25)
        rows.append({
            'sensor_id': f'S{sensor_id:03d}',
            'observed_temperature_c': observed,
            'geometry': Point(px, py),
        })
    return gpd.GeoDataFrame(rows, crs=grid_gdf.crs)

sensors = sample_sensors(grid, CFG)

sensor_joined = gpd.sjoin(
    sensors,
    grid[['cell_id','temperature_c','vegetation','building_density','geometry']],
    how='left',
    predicate='within',
)

display(sensor_joined.head())

,sensor_id,observed_temperature_c,geometry,index_right,cell_id,temperature_c,vegetation,building_density
0,S000,28.989055,POINT (15502.145 12529.172),279,R12_C15,28.524471,0.184610,0.140004
1,S001,30.449326,POINT (2655.661 4275.579),90,R04_C02,30.347663,0.215930,0.218647
2,S002,32.590768,POINT (10279.739 626.539),10,R00_C10,32.203264,0.208249,0.103932
3,S003,32.198515,POINT (4325.201 10894.581),224,R10_C04,32.466721,0.610702,0.206125
4,S004,28.321011,POINT (14326.687 5681.751),124,R05_C14,28.310417,0.216603,0.522180


## 7.1 Spatial join data-quality checks

Geospatial ETL needs the same discipline as ordinary ETL.

Check:

- row counts before and after join;
- unmatched geometries;
- duplicate matches;
- null geometries;
- CRS consistency.

In [12]:
print('Sensor rows before join:', len(sensors))
print('Rows after join:', len(sensor_joined))
print('Unmatched sensors:', sensor_joined['cell_id'].isna().sum())
print('Duplicate sensor matches:', sensor_joined['sensor_id'].duplicated().sum())
print('Same CRS:', sensors.crs == grid.crs)

Sensor rows before join: 85
Rows after join: 85
Unmatched sensors: 0
Duplicate sensor matches: 0
Same CRS: True


# 8. Buffers

A buffer around a point $p$ with radius $r$ is

$$
B(p,r)=\{x:d(x,p)\le r\}.
$$

Typical applications:

- buildings within 500 m of a highway;
- residents within 1 km of a cooling centre;
- habitat within 2 km of a development site.

In [13]:
example_sensor = sensors.iloc[[0]].copy()
buffer_1500 = example_sensor.copy()
buffer_1500['geometry'] = buffer_1500.geometry.buffer(1500)
buffer_1500['cell_id'] = '1.5 km buffer'

base_src = geodata_source(grid)
buffer_src = geodata_source(buffer_1500)

p = figure(width=800, height=450, title='Example: 1.5 km buffer around one sensor', match_aspect=True,
           tools='pan,wheel_zoom,box_zoom,reset')
p.patches('xs','ys',source=base_src,fill_alpha=0.07,line_alpha=0.25)
p.patches('xs','ys',source=buffer_src,fill_alpha=0.20,line_width=2)
p.scatter([example_sensor.geometry.iloc[0].x],[example_sensor.geometry.iloc[0].y],size=12,marker='circle')
show(p)

# 9. Nearest-neighbour distance

In projected coordinates, Euclidean distance is

$$
d_{ij}=\sqrt{(x_i-x_j)^2+(y_i-y_j)^2}.
$$

But the correct distance depends on the domain:

- Euclidean distance — physical proximity;
- great-circle distance — long geographic distances;
- network distance — travel along roads;
- travel time — accessibility;
- cost distance — terrain/friction-aware movement.

In [43]:
sensor_xy = np.column_stack([sensors.geometry.x.to_numpy(), sensors.geometry.y.to_numpy()])
distance_matrix = cdist(sensor_xy, sensor_xy)
np.fill_diagonal(distance_matrix, np.inf)
nearest_idx = distance_matrix.argmin(axis=1)
nearest_dist = distance_matrix.min(axis=1)

nearest_table = pd.DataFrame({
    'sensor_id': sensors['sensor_id'].to_numpy(),
    'nearest_sensor_id': sensors.iloc[nearest_idx]['sensor_id'].to_numpy(),
    'nearest_distance_m': nearest_dist,
}).sort_values('nearest_distance_m')
display(nearest_table.head(10))

,sensor_id,nearest_sensor_id,nearest_distance_m
29,S029,S019,336.681682
19,S019,S029,336.681682
42,S042,S068,482.115533
68,S068,S042,482.115533
8,S008,S003,487.813624
3,S003,S008,487.813624
39,S039,S002,624.388820
2,S002,S039,624.388820
71,S071,S033,660.125808
33,S033,S071,660.125808


# 10. Spatial weights and neighbourhood graphs

Spatial statistical models need to formalise who is a neighbour of whom.

A spatial weights matrix is

$$
W=(w_{ij}).
$$

A common rule is contiguity:

$$
w_{ij}=1
$$

when polygons $i$ and $j$ share a boundary.

### Rook vs Queen

- **Rook:** shared edge;
- **Queen:** shared edge or corner.

The choice of $W$ is a modelling assumption, not a trivial implementation detail.

In [44]:
from libpysal.weights import Queen, Rook

w_queen = Queen.from_dataframe(grid, ids=grid['cell_id'].tolist(), use_index=False)
w_queen.transform = 'r'

w_rook = Rook.from_dataframe(grid, ids=grid['cell_id'].tolist(), use_index=False)
w_rook.transform = 'r'

summary = pd.DataFrame({
    'weights': ['Queen','Rook'],
    'mean_neighbours': [
        np.mean(list(w_queen.cardinalities.values())),
        np.mean(list(w_rook.cardinalities.values())),
    ],
    'min_neighbours': [min(w_queen.cardinalities.values()), min(w_rook.cardinalities.values())],
    'max_neighbours': [max(w_queen.cardinalities.values()), max(w_rook.cardinalities.values())],
})
display(summary.round(3))

,weights,mean_neighbours,min_neighbours,max_neighbours
0,Queen,7.312,3,8
1,Rook,3.766,2,4


## 10.1 Row standardisation

If an area has four neighbours, an unstandardised lag is

$$
(Wy)_i=y_{j_1}+y_{j_2}+y_{j_3}+y_{j_4}.
$$

After row standardisation,

$$
(Wy)_i=\frac{y_{j_1}+y_{j_2}+y_{j_3}+y_{j_4}}{4}.
$$

So $Wy$ behaves like a neighbourhood average.

In [45]:
def spatial_lag_from_weights(values: pd.Series, weights, ids: pd.Series) -> np.ndarray:
    lookup = dict(zip(ids, values.to_numpy()))
    lagged = []
    for unit_id in ids:
        neigh = weights.neighbors[unit_id]
        wts = weights.weights[unit_id]
        lagged.append(sum(w*lookup[n] for n,w in zip(neigh,wts)))
    return np.asarray(lagged)

grid['temperature_spatial_lag'] = spatial_lag_from_weights(grid['temperature_c'], w_queen, grid['cell_id'])
display(grid[['cell_id','temperature_c','temperature_spatial_lag']].head())

,cell_id,temperature_c,temperature_spatial_lag
0,R00_C00,29.782678,30.783875
1,R00_C01,31.396344,30.930715
2,R00_C02,31.833937,31.687575
3,R00_C03,31.773684,32.545322
4,R00_C04,32.896493,32.652367


# 11. Moran scatter plot

Standardise the outcome:

$$
z_i=\frac{x_i-\bar{x}}{s_x}.
$$

Plot $z_i$ against $Wz_i$.

Quadrants:

- High–High;
- Low–Low;
- High–Low spatial outlier;
- Low–High spatial outlier.

In [46]:
temp = grid['temperature_c'].to_numpy()
z_temp = (temp - temp.mean()) / temp.std(ddof=0)
grid['z_temperature'] = z_temp
grid['lag_z_temperature'] = spatial_lag_from_weights(pd.Series(z_temp), w_queen, grid['cell_id'])

p = figure(width=720,height=470,title='Moran scatter plot',x_axis_label='Standardised temperature z',
           y_axis_label='Spatial lag Wz',tools='pan,wheel_zoom,box_zoom,reset,save')
src = ColumnDataSource(grid[['cell_id','z_temperature','lag_z_temperature']])
r = p.scatter('z_temperature','lag_z_temperature',source=src,size=7,alpha=0.65)
p.add_tools(HoverTool(renderers=[r],tooltips=[('cell','@cell_id'),('z','@z_temperature{0.000}'),('Wz','@lag_z_temperature{0.000}')]))
p.add_layout(Span(location=0,dimension='height',line_dash='dashed'))
p.add_layout(Span(location=0,dimension='width',line_dash='dashed'))
coef = np.polyfit(grid['z_temperature'], grid['lag_z_temperature'], 1)
xline = np.linspace(grid['z_temperature'].min(), grid['z_temperature'].max(), 100)
p.line(xline, coef[0]*xline+coef[1], line_width=2)
show(p)

# 12. Global Moran's $I$

Moran's $I$ measures global spatial autocorrelation:

$$
I=
\frac{n}{\sum_i\sum_j w_{ij}}
\frac{\sum_i\sum_j w_{ij}(x_i-\bar{x})(x_j-\bar{x})}{\sum_i(x_i-\bar{x})^2}.
$$

Broad interpretation:

- $I>0$: similar values cluster;
- $I<0$: neighbouring values tend to differ;
- $I\approx0$: weak global spatial pattern.

A permutation test shuffles values over locations to approximate the null distribution under spatial randomness.

In [18]:
from esda.moran import Moran

mi_q = Moran(grid['temperature_c'].to_numpy(), w_queen, permutations=999)
mi_r = Moran(grid['temperature_c'].to_numpy(), w_rook, permutations=999)

moran_table = pd.DataFrame({
    'weights':['Queen','Rook'],
    'Moran_I':[mi_q.I, mi_r.I],
    'expected_I':[mi_q.EI, mi_r.EI],
    'p_sim':[mi_q.p_sim, mi_r.p_sim],
})
display(moran_table.round(4))

,weights,Moran_I,expected_I,p_sim
0,Queen,0.8263,-0.0033,0.001
1,Rook,0.8404,-0.0033,0.001


### Interpretation checkpoint

A significant positive Moran's $I$ means temperature is spatially clustered.

It does **not** prove that proximity causes temperature. It only demonstrates spatial dependence.

# 13. Local Moran / LISA

Global Moran's $I$ answers:

> Does the whole study area exhibit spatial clustering?

Local Moran's $I$ asks:

> Where are the local clusters and outliers?

Useful classes:

| Class | Meaning |
|---|---|
| High–High | hot area surrounded by hot areas |
| Low–Low | cool area surrounded by cool areas |
| High–Low | hot spatial outlier |
| Low–High | cool spatial outlier |

In [19]:
from esda.moran import Moran_Local

lisa = Moran_Local(grid['temperature_c'].to_numpy(), w_queen, permutations=999)
qmap = {1:'High-High',2:'Low-High',3:'Low-Low',4:'High-Low'}
grid['lisa_quadrant'] = [qmap[q] for q in lisa.q]
grid['lisa_p'] = lisa.p_sim
grid['lisa_cluster'] = np.where(grid['lisa_p'] < 0.05, grid['lisa_quadrant'], 'Not significant')
display(grid['lisa_cluster'].value_counts().to_frame('count'))

,count
lisa_cluster,
Not significant,119
High-High,100
Low-Low,84
Low-High,3
High-Low,2


In [47]:
LISA_COLORS = {
    'High-High': Category10[5][3],
    'Low-Low': Category10[5][0],
    'High-Low': Category10[5][1],
    'Low-High': Category10[5][2],
    'Not significant': Category10[5][4],
}
plot_df = grid.copy()
plot_df['cluster_color'] = plot_df['lisa_cluster'].map(LISA_COLORS)
src = geodata_source(plot_df, ['lisa_cluster','lisa_p','cluster_color'])

p = figure(width=820,height=470,title='Local Moran / LISA clusters',match_aspect=True,
           tools='pan,wheel_zoom,box_zoom,reset,save')
r = p.patches('xs','ys',source=src,fill_color='cluster_color',fill_alpha=0.88,line_alpha=0.30)
p.add_tools(HoverTool(renderers=[r],tooltips=[('cell','@cell_id'),('cluster','@lisa_cluster'),('p','@lisa_p{0.0000}')]))
show(p)

# 14. Hazard vs exposure

A sustainability decision is often not based on hazard alone.

$$
\text{hazard}\neq\text{exposure}\neq\text{risk}.
$$

A hot but nearly uninhabited location may be less urgent than a slightly cooler but densely populated location.

For teaching, define

$$
E_i=T_i\log(1+P_i).
$$

This is an illustrative exposure index, not a validated epidemiological risk score.

In [48]:
show(choropleth(grid, 'heat_exposure_index', 'Illustrative population-weighted heat exposure', palette=Turbo256))

# 15. OLS baseline

We first fit a conventional model:

$$
T_i=\beta_0+\beta_1V_i+\beta_2B_i+\beta_3R_i+\beta_4D_i+\epsilon_i.
$$

This follows a sound modelling sequence:

1. build a simple baseline;
2. inspect predictive performance;
3. inspect residuals;
4. test residual spatial dependence;
5. only then justify a spatial model.

In [49]:
FEATURES = ['vegetation','building_density','road_density','distance_to_coast_km']
X = grid[FEATURES]
y_target = grid['temperature_c']

ols = LinearRegression().fit(X, y_target)
grid['ols_prediction'] = ols.predict(X)
grid['ols_residual'] = y_target - grid['ols_prediction']

coef_table = pd.DataFrame({'feature':FEATURES,'coefficient':ols.coef_})
print('Intercept:', round(float(ols.intercept_),4))
display(coef_table.round(4))

Intercept: 29.4869


,feature,coefficient
0,vegetation,0.2056
1,building_density,2.5073
2,road_density,-0.5646
3,distance_to_coast_km,-0.0136


In [23]:
def regression_metrics(y_true, y_pred) -> pd.Series:
    return pd.Series({
        'MAE': mean_absolute_error(y_true,y_pred),
        'RMSE': mean_squared_error(y_true,y_pred)**0.5,
        'R2': r2_score(y_true,y_pred),
    })

display(regression_metrics(grid['temperature_c'],grid['ols_prediction']).to_frame('OLS').round(4))

,OLS
MAE,1.3382
RMSE,1.6285
R2,0.0269


## 15.1 Residual spatial autocorrelation

A high $R^2$ does not guarantee a well-specified spatial model.

If residuals in neighbouring areas are similar, the model is systematically under- or over-predicting whole regions.

We therefore apply Moran's $I$ to OLS residuals.

In [50]:
mi_resid = Moran(grid['ols_residual'].to_numpy(), w_queen, permutations=999)
print(f"Residual Moran's I: {mi_resid.I:.4f}")
print(f"Permutation p-value: {mi_resid.p_sim:.4f}")
show(choropleth(grid, 'ols_residual', 'OLS residuals — spatial structure left unexplained', palette=Turbo256))

Residual Moran's I: 0.8260
Permutation p-value: 0.0010


If residual spatial autocorrelation remains significant, plausible explanations include:

- omitted spatially varying variables;
- spillover/diffusion effects;
- spatial measurement error;
- wrong functional form;
- wrong geographic scale.

# 16. Spatial lag model (SAR)

A spatial lag model is

$$
y=\rho Wy+X\beta+\epsilon.
$$

The term $Wy$ is the neighbouring outcome average (after row standardisation).

The parameter $\rho$ captures response dependence across neighbouring areas.

In [25]:
from spreg import ML_Lag

y_sp = grid['temperature_c'].to_numpy().reshape(-1,1)
X_sp = grid[FEATURES].to_numpy()

sar = ML_Lag(y=y_sp, x=X_sp, w=w_queen, name_y='temperature_c', name_x=FEATURES)
print(sar.summary[:6000])

ML_Lag
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: MAXIMUM LIKELIHOOD SPATIAL LAG (METHOD = FULL)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :temperature_c                Number of Observations:         308
Mean dependent var  :     29.9679                Number of Variables   :           6
S.D. dependent var  :      1.6535                Degrees of Freedom    :         302
Pseudo R-squared    :      0.8384
Spatial Pseudo R-squared:  0.0045
Log likelihood      :   -349.2930
Sigma-square ML     :      0.4579                Akaike info criterion :     710.586
S.E of regression   :      0.6767                Schwarz criterion     :     732.967

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------

# 17. Spatial error model (SEM)

A spatial error model is

$$
y=X\beta+u,
$$

with

$$
u=\lambda Wu+\epsilon.
$$

Here the dependence is interpreted as spatially structured omitted factors rather than direct outcome spillover.

In [26]:
from spreg import ML_Error

sem = ML_Error(y=y_sp, x=X_sp, w=w_queen, name_y='temperature_c', name_x=FEATURES)
print(sem.summary[:6000])

/home/anirban/polyglot-jupyter/.venv/lib/python3.11/site-packages/spreg/ml_error.py:184: RuntimeWarning: Method 'bounded' does not support relative tolerance in x; defaulting to absolute tolerance.
  res = minimize_scalar(


ML_Error
REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: ML SPATIAL ERROR (METHOD = full)
------------------------------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :temperature_c                Number of Observations:         308
Mean dependent var  :     29.9679                Number of Variables   :           5
S.D. dependent var  :      1.6535                Degrees of Freedom    :         303
Pseudo R-squared    :      0.0196
Log likelihood      :   -346.4256
Sigma-square ML     :      0.4481                Akaike info criterion :     702.851
S.E of regression   :      0.6694                Schwarz criterion     :     721.502

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
----------------------------------------------------------------------------------

# 18. Comparing OLS, SAR and SEM

Do not select a spatial model by one number alone.

Consider:

- AIC / likelihood;
- residual autocorrelation;
- predictive performance;
- interpretability;
- scientific mechanism;
- robustness to the definition of $W$.

In [27]:
model_comparison = pd.DataFrame([
    {'model':'OLS','AIC':np.nan,'log_likelihood':np.nan,'residual_Moran_I':mi_resid.I,'residual_Moran_p':mi_resid.p_sim},
    {'model':'SAR','AIC':sar.aic,'log_likelihood':sar.logll,'residual_Moran_I':np.nan,'residual_Moran_p':np.nan},
    {'model':'SEM','AIC':sem.aic,'log_likelihood':sem.logll,'residual_Moran_I':np.nan,'residual_Moran_p':np.nan},
])
display(model_comparison.round(4))

,model,AIC,log_likelihood,residual_Moran_I,residual_Moran_p
0,OLS,NaN,NaN,0.826,0.001
1,SAR,710.5861,-349.2930,NaN,NaN
2,SEM,702.8512,-346.4256,NaN,NaN


# 19. Point-pattern analysis with KDE

Point processes appear in many applications:

- dengue cases;
- accidents;
- fires;
- trees;
- wildlife sightings;
- EV chargers.

Kernel density estimation creates a smoothed intensity surface.

$$
\hat\lambda(s)=\frac{1}{nh^2}\sum_{i=1}^nK\left(\frac{\|s-s_i\|}{h}\right).
$$

The bandwidth $h$ controls smoothness.

In [51]:
sensor_x = sensors.geometry.x.to_numpy()
sensor_y = sensors.geometry.y.to_numpy()
xy = np.vstack([sensor_x,sensor_y])
kde = gaussian_kde(xy)

xmin,ymin,xmax,ymax = grid.total_bounds
gx = np.linspace(xmin,xmax,140)
gy = np.linspace(ymin,ymax,100)
xx,yy = np.meshgrid(gx,gy)
positions = np.vstack([xx.ravel(),yy.ravel()])
density = kde(positions).reshape(xx.shape)

p = figure(width=820,height=470,title='Kernel density surface of sensor locations',x_range=(xmin,xmax),y_range=(ymin,ymax),
           match_aspect=True,tools='pan,wheel_zoom,box_zoom,reset,save')
mapper = LinearColorMapper(palette=Viridis256,low=float(density.min()),high=float(density.max()))
p.image(image=[density],x=xmin,y=ymin,dw=xmax-xmin,dh=ymax-ymin,color_mapper=mapper,alpha=0.85)
p.scatter(sensor_x,sensor_y,size=5,alpha=0.70)
p.add_layout(ColorBar(color_mapper=mapper),'right')
show(p)

### KDE bandwidth intuition

Small bandwidth:

- more local detail;
- noisier surface;
- many small hotspots.

Large bandwidth:

- smoother surface;
- fewer broad hotspots;
- local structure may be hidden.

This is a spatial form of the bias–variance trade-off.

# 20. Spatial interpolation with inverse-distance weighting (IDW)

Environmental monitoring frequently has sparse sensors.

For an unobserved point $s_0$,

$$
\hat z(s_0)=\frac{\sum_iw_iz(s_i)}{\sum_iw_i},
\qquad
w_i=\frac{1}{d(s_0,s_i)^p}.
$$

The power parameter $p$ determines how quickly influence decreases with distance.

In [60]:
class IDWInterpolator:
    def __init__(self, power: float = 2.0, epsilon: float = 1e-9):
        self.power = power
        self.epsilon = epsilon
        self.points_ = None
        self.values_ = None

    def fit(self, points: np.ndarray, values: np.ndarray):
        self.points_ = np.asarray(points,dtype=float)
        self.values_ = np.asarray(values,dtype=float)
        return self

    def predict(self, query_points: np.ndarray) -> np.ndarray:
        if self.points_ is None or self.values_ is None:
            raise RuntimeError('Call fit() before predict().')
        query_points = np.asarray(query_points,dtype=float)
        distances = cdist(query_points,self.points_)
        exact = distances < self.epsilon
        safe = np.maximum(distances,self.epsilon)
        weights = 1.0 / np.power(safe,self.power)
        preds = (weights @ self.values_) / weights.sum(axis=1)
        for i in range(len(query_points)):
            if exact[i].any():
                preds[i] = self.values_[np.argmax(exact[i])]
        return preds

sensor_points = np.column_stack([sensors.geometry.x,sensors.geometry.y])
sensor_values = sensors['observed_temperature_c'].to_numpy()
grid_centroids = np.column_stack([grid.geometry.centroid.x,grid.geometry.centroid.y])

idw = IDWInterpolator(power=2.0).fit(sensor_points,sensor_values)
grid['idw_temperature'] = idw.predict(grid_centroids)
display(regression_metrics(grid['temperature_c'],grid['idw_temperature']).to_frame('IDW').round(4))

,IDW
MAE,0.6499
RMSE,0.8462
R2,0.7373


In [30]:
show(row(
    choropleth(grid,'temperature_c','True simulated temperature',palette=Turbo256,width=540,height=360),
    choropleth(grid,'idw_temperature','IDW interpolation',palette=Turbo256,width=540,height=360),
))

## 20.1 IDW power experiment

A useful sensitivity analysis varies $p$.

Low $p$ spreads influence broadly. High $p$ makes the estimate dominated by the nearest sensor.

In [53]:
rows=[]
for pwr in [0.5,1,2,4,8]:
    pred = IDWInterpolator(power=pwr).fit(sensor_points,sensor_values).predict(grid_centroids)
    m = regression_metrics(grid['temperature_c'],pred)
    rows.append({'power':pwr,**m.to_dict()})
idw_sensitivity = pd.DataFrame(rows)
display(idw_sensitivity.round(4))

p = figure(width=720,height=420,title='IDW sensitivity to distance-decay power',x_axis_label='Power p',y_axis_label='RMSE')
p.line(idw_sensitivity['power'],idw_sensitivity['RMSE'],line_width=2)
p.scatter(idw_sensitivity['power'],idw_sensitivity['RMSE'],size=9)
show(p)

,power,MAE,RMSE,R2
0,0.5,1.1647,1.4063,0.2743
1,1.0,0.9395,1.1419,0.5215
2,2.0,0.6499,0.8462,0.7373
3,4.0,0.6011,0.7872,0.7726
4,8.0,0.6159,0.8099,0.7593


# 21. Empirical variogram

Geostatistics studies how dissimilarity changes with distance.

$$
\gamma(h)=\frac12\operatorname{Var}[Z(s+h)-Z(s)].
$$

Empirically,

$$
\hat\gamma(h)=\frac{1}{2|N(h)|}\sum_{(i,j)\in N(h)}[z(s_i)-z(s_j)]^2.
$$

Common concepts:

- **nugget** — very short-scale variation / measurement noise;
- **sill** — long-distance semivariance plateau;
- **range** — distance beyond which spatial correlation is weak.

In [54]:
def empirical_variogram(points, values, n_bins=15, max_distance=None):
    points = np.asarray(points,dtype=float)
    values = np.asarray(values,dtype=float)
    dmat = cdist(points,points)
    iu = np.triu_indices(len(points),k=1)
    d = dmat[iu]
    semiv = 0.5*(values[iu[0]]-values[iu[1]])**2
    if max_distance is None:
        max_distance = np.quantile(d,0.85)
    mask = d <= max_distance
    d, semiv = d[mask], semiv[mask]
    edges = np.linspace(0,max_distance,n_bins+1)
    bin_id = np.digitize(d,edges)-1
    rows=[]
    for b in range(n_bins):
        m = bin_id == b
        if m.any():
            rows.append({'distance_m':d[m].mean(),'semivariance':semiv[m].mean(),'pair_count':int(m.sum())})
    return pd.DataFrame(rows)

variogram_df = empirical_variogram(sensor_points,sensor_values,n_bins=16)
display(variogram_df.head())

,distance_m,semivariance,pair_count
0,705.996417,0.696016,19
1,1437.553976,0.587602,84
2,2341.131036,0.784287,141
3,3295.285699,0.891071,182
4,4207.092725,0.972907,194


In [33]:
p = figure(width=760,height=440,title='Empirical semivariogram',x_axis_label='Distance between sensors (m)',y_axis_label='Semivariance',
           tools='pan,wheel_zoom,box_zoom,reset,save')
src = ColumnDataSource(variogram_df)
r = p.scatter('distance_m','semivariance',source=src,size=9)
p.line('distance_m','semivariance',source=src,line_width=2)
p.add_tools(HoverTool(renderers=[r],tooltips=[('distance','@distance_m{0} m'),('semivariance','@semivariance{0.000}'),('pairs','@pair_count')]))
show(p)

### Moran vs variogram intuition

Moran-style approaches are especially natural for **areal data** represented by adjacency graphs.

Variograms are especially natural for **continuous spatial fields** measured at irregular sensor locations.

Both study spatial dependence, but with different mathematical representations.

# 22. Spatial cross-validation

Random train/test splitting can leak geographic similarity.

If a training point is very close to a test point, evaluation partly rewards local interpolation rather than genuine geographical generalisation.

Spatial block CV holds out whole geographic regions.

In [55]:
def assign_spatial_blocks(gdf, n_x_blocks=4, n_y_blocks=3):
    cent = gdf.geometry.centroid
    xb = pd.cut(cent.x,bins=n_x_blocks,labels=False,include_lowest=True)
    yb = pd.cut(cent.y,bins=n_y_blocks,labels=False,include_lowest=True)
    return (yb*n_x_blocks+xb).astype(int)

grid['spatial_block'] = assign_spatial_blocks(grid)
display(grid['spatial_block'].value_counts().sort_index().to_frame('cells'))

,cells
spatial_block,
0,30
1,25
2,25
3,30
4,24
5,20
6,20
7,24
8,30


In [35]:
def spatial_block_cv(gdf, features, target):
    rows=[]
    for block in sorted(gdf['spatial_block'].unique()):
        train = gdf[gdf['spatial_block'] != block]
        test = gdf[gdf['spatial_block'] == block]
        model = LinearRegression().fit(train[list(features)],train[target])
        pred = model.predict(test[list(features)])
        rows.append({
            'held_out_block':block,
            'n_test':len(test),
            'MAE':mean_absolute_error(test[target],pred),
            'RMSE':mean_squared_error(test[target],pred)**0.5,
            'R2':r2_score(test[target],pred),
        })
    return pd.DataFrame(rows)

block_cv = spatial_block_cv(grid,FEATURES,'temperature_c')
display(block_cv.round(4))

,held_out_block,n_test,MAE,RMSE,R2
0,0,30,1.8142,2.1330,-1.6204
1,1,25,2.2923,2.4932,-6.2537
2,2,25,0.8773,1.1053,-0.4083
3,3,30,1.5850,1.9300,-2.1625
4,4,24,0.9218,1.1505,-0.1673
5,5,20,2.0239,2.1177,-9.8095
6,6,20,1.1108,1.4012,-0.2885
7,7,24,2.9926,3.1241,-13.0316
8,8,30,1.2537,1.4951,-2.3006
9,9,25,1.1957,1.3444,-0.8244


## 22.1 Compare random CV with spatial CV

A random split often produces a more optimistic estimate because geographically similar observations appear in both training and validation data.

In [36]:
random_cv = KFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
rmse_random = (-cross_val_score(LinearRegression(),X,y_target,cv=random_cv,scoring='neg_mean_squared_error'))**0.5

comparison_cv = pd.DataFrame({
    'scheme':['Random 5-fold','Spatial block CV'],
    'mean_RMSE':[rmse_random.mean(),block_cv['RMSE'].mean()],
    'std_RMSE':[rmse_random.std(ddof=1),block_cv['RMSE'].std(ddof=1)],
})
display(comparison_cv.round(4))

,scheme,mean_RMSE,std_RMSE
0,Random 5-fold,1.6452,0.0386
1,Spatial block CV,1.7748,0.6086


# 23. Spatial weights as a graph

The adjacency matrix $W$ is also a graph:

- polygons are nodes;
- adjacency relationships are edges.

This connects spatial statistics to graph algorithms.

In [56]:
G = nx.Graph()
for cell_id in grid['cell_id']:
    G.add_node(cell_id)
for cell_id, neighbours in w_queen.neighbors.items():
    for neighbour in neighbours:
        G.add_edge(cell_id,neighbour)

print('Nodes:',G.number_of_nodes())
print('Edges:',G.number_of_edges())
print('Connected components:',nx.number_connected_components(G))

degree_df = pd.DataFrame({'cell_id':list(dict(G.degree()).keys()),'degree':list(dict(G.degree()).values())}).sort_values('degree',ascending=False)
display(degree_df.head())

Nodes: 308
Edges: 1126
Connected components: 1


,cell_id,degree
284,R12_C20,8
283,R12_C19,8
282,R12_C18,8
281,R12_C17,8
280,R12_C16,8


# 24. Spatiotemporal extension

Many sustainability variables are functions of space **and** time:

$$
Z(s,t).
$$

Examples:

- PM2.5 by sensor and hour;
- rainfall by station and day;
- traffic by road segment and minute;
- temperature by district and hour.

A simple decomposition is

$$
T(s,t)=\mu+S(s)+H(t)+\epsilon(s,t).
$$

In [57]:
selected = grid.sample(n=6,random_state=RANDOM_STATE).copy()
rows=[]
for _,rec in selected.iterrows():
    for hour in range(24):
        diurnal = 2.8*np.sin(2*np.pi*(hour-8)/24)
        rows.append({'cell_id':rec['cell_id'],'hour':hour,'temperature_c':rec['temperature_c']+diurnal})
hourly = pd.DataFrame(rows)

p = figure(width=800,height=450,title='Spatiotemporal example: hourly temperature by selected cell',x_axis_label='Hour',y_axis_label='Temperature (°C)',
           tools='pan,wheel_zoom,box_zoom,reset,save')
for cell_id,group in hourly.groupby('cell_id'):
    p.line(group['hour'],group['temperature_c'],line_width=2,legend_label=cell_id)
p.legend.location='top_left'
p.legend.click_policy='hide'
show(p)

# 25. Raster thinking

Raster data represent space as cells or pixels:

$$
R=\begin{bmatrix}
r_{11}&r_{12}&\cdots\\
r_{21}&r_{22}&\cdots\\
\vdots&\vdots&\ddots
\end{bmatrix}.
$$

Typical rasters:

- satellite imagery;
- elevation;
- rainfall;
- land-surface temperature;
- NDVI;
- land-cover classification.

Our regular polygon grid can be reshaped into a raster-like matrix.

In [58]:
temperature_raster = grid.pivot(index='row',columns='col',values='temperature_c').sort_index().to_numpy()

p = figure(width=800,height=420,title='Raster-like view of the temperature field',x_range=(0,CFG.n_cols),y_range=(0,CFG.n_rows),
           tools='pan,wheel_zoom,box_zoom,reset,save')
mapper = LinearColorMapper(palette=Turbo256,low=float(np.nanmin(temperature_raster)),high=float(np.nanmax(temperature_raster)))
p.image(image=[temperature_raster],x=0,y=0,dw=CFG.n_cols,dh=CFG.n_rows,color_mapper=mapper)
p.add_layout(ColorBar(color_mapper=mapper),'right')
show(p)

## Vector vs raster

| Aspect | Vector | Raster |
|---|---|---|
| Best suited for | discrete objects | continuous fields / imagery |
| Examples | roads, parcels, districts | temperature, elevation |
| Core operations | join, overlay, buffer | map algebra, convolution |
| Storage | coordinates + geometry | pixel/cell grid |
| Scaling challenge | many complex geometries | huge number of pixels |

# 26. Spatial indexing and algorithmic complexity

A naive all-pairs geometry comparison requires

$$
\Theta(n^2)
$$

candidate comparisons.

At large $n$, this is unacceptable.

Spatial indexes such as R-trees, quadtrees and k-d trees prune impossible candidates before exact geometry operations.

That is why spatial indexing is a data-engineering concern as much as a GIS concern.

In [59]:
spatial_index = grid.sindex
print(type(spatial_index))
print('Spatial-index size:',spatial_index.size)

<class 'geopandas.sindex.SpatialIndex'>
Spatial-index size: 308


# 27. Scale and the Modifiable Areal Unit Problem (MAUP)

Results can change when data are aggregated into different geographic units.

For example, the same point observations could be summarised by:

- 500 m grid cells;
- 1 km grid cells;
- planning areas;
- administrative regions.

Means, correlations, hotspot boundaries and regression coefficients may change.

This is the **Modifiable Areal Unit Problem**.

A critical geospatial question is therefore:

> **At what spatial scale should the phenomenon be modelled?**

# 28. Methodical end-to-end workflow

A strong DSS5103-style workflow is:

```text
1. Define the spatial decision question
        ↓
2. Choose the spatial unit and scale
        ↓
3. Load vector/raster data
        ↓
4. Validate geometry and CRS
        ↓
5. Clean invalid/missing geometry
        ↓
6. Reproject where appropriate
        ↓
7. Spatial joins / overlays / buffers
        ↓
8. Engineer spatial features
        ↓
9. Visualise and perform ESDA
        ↓
10. Define spatial weights / covariance
        ↓
11. Test spatial dependence
        ↓
12. Fit a conventional baseline
        ↓
13. Diagnose residual spatial dependence
        ↓
14. Fit spatial model when justified
        ↓
15. Validate spatially
        ↓
16. Quantify exposure and uncertainty
        ↓
17. Translate results into a sustainability decision
```

# 29. Common mistakes

## 1. Ignoring CRS
Distance and area calculations can become meaningless.

## 2. Treating a map as statistical proof
Visual clustering does not automatically imply statistical significance.

## 3. Assuming independence
Neighbouring observations can share latent structure.

## 4. Choosing $W$ arbitrarily
Queen, Rook, kNN and distance-band weights encode different assumptions.

## 5. Confusing dependence with causality
Spatial regression does not automatically identify causal effects.

## 6. Random CV leakage
Nearby train/test points can exaggerate predictive performance.

## 7. Ignoring scale / MAUP
Aggregation can materially alter results.

## 8. Forgetting edge effects
Boundary regions have fewer possible neighbours and point-pattern KDE is biased near study-area boundaries.

# 30. Guided student experiments

## Exercise A — Queen vs Rook

Compare:

- mean neighbour count;
- Moran's $I$;
- LISA clusters.

Explain why changing neighbourhood semantics changes the statistical model.

## Exercise B — spatial signal strength

Change `spatial_noise_strength` to:

```text
0.0, 0.5, 1.0, 2.5, 4.0
```

Re-run OLS and residual Moran's $I$.

## Exercise C — sensor scarcity

Use:

```text
20, 40, 80, 160 sensors
```

Compare IDW RMSE.

## Exercise D — IDW power

Try:

```text
0.5, 1, 2, 4, 8
```

Explain why high power approaches nearest-neighbour interpolation.

## Exercise E — omitted variables

Remove `vegetation` from the OLS feature list.

Compare:

- coefficient changes;
- RMSE;
- residual Moran's $I$.

## Exercise F — evaluation design

Compare random CV with spatial block CV.

Explain which estimate better reflects deployment to a new region.

## Exercise G — hazard vs exposure

Rank cells by temperature and then by exposure index.

Why do the rankings differ?

# 31. Advanced extensions

After mastering this notebook, natural extensions are:

### Geostatistics

- fitted variogram models;
- ordinary kriging;
- universal kriging;
- Gaussian processes;
- prediction uncertainty surfaces.

### Bayesian spatial modelling

- CAR models;
- SAR priors;
- hierarchical spatial models;
- disease mapping.

### Remote sensing

- multispectral imagery;
- NDVI / NDBI;
- land-cover classification;
- semantic segmentation.

### Spatial machine learning

- coordinate and neighbourhood feature engineering;
- geographically weighted regression;
- spatial boosting;
- graph neural networks.

### Spatial databases and large-scale engineering

- PostGIS;
- GeoParquet;
- cloud-optimised GeoTIFF;
- Dask-GeoPandas;
- Apache Sedona;
- partitioning and spatial indexes.

### Spatiotemporal modelling

- dynamic spatial regression;
- state-space models;
- Gaussian processes over space-time;
- graph-temporal neural networks.

# 32. Concept map and final takeaway

The intellectual progression is:

$$
\boxed{
\text{Where?}
\rightarrow
\text{What is nearby?}
\rightarrow
\text{What pattern exists?}
\rightarrow
\text{Is it statistically meaningful?}
\rightarrow
\text{What explains it?}
\rightarrow
\text{Can we predict elsewhere?}
\rightarrow
\text{What decision follows?}
}
$$

Technically:

```text
Geometry
   ↓
CRS
   ↓
Vector / Raster
   ↓
Spatial operations
   ↓
Spatial weights / covariance
   ↓
Autocorrelation
   ↓
Spatial regression / interpolation
   ↓
Spatial validation
   ↓
Sustainability decision
```

The deepest lesson is:

> **Location is not metadata. Location is part of the statistical structure.**

A strong geospatial workflow explicitly models geometry, distance, neighbourhood, dependence and scale instead of treating coordinates as just two extra columns.